# Customer Churn Prediction - CodSoft Task 3

**Dataset:** Bank Customer Churn Dataset (`Churn_Modelling.csv`)
**Source:** [Kaggle - Bank Customer Churn Modeling](https://www.kaggle.com/datasets/barelydedicated/bank-customer-churn-modeling)

This notebook demonstrates an end-to-end Machine Learning pipeline using Scikit-Learn:
1. Exploratory Data Analysis & Preprocessing
2. Stratified Train-Test Split
3. Scikit-Learn `ColumnTransformer` + `Pipeline` with `LogisticRegression`
4. Model Evaluation (Confusion Matrix, Precision, Recall, F1-Score, ROC-AUC)
5. Exporting `churn_model.pkl` for deployment with Streamlit

### Step 1: Upload Dataset (For Google Colab)
Run this cell in Google Colab to upload `Churn_Modelling.csv`. If running locally, make sure `Churn_Modelling.csv` is in the same folder as this notebook.

In [ ]:
# Step 1: Upload dataset when running on Google Colab
try:
    from google.colab import files
    print("Please select and upload Churn_Modelling.csv:")
    uploaded = files.upload()
except ImportError:
    print("Running locally outside Google Colab. Looking for Churn_Modelling.csv in workspace.")

### Step 2: Import Libraries

In [ ]:
# Step 2: Import essential data science and ML libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)
import joblib

# Set plot appearance
sns.set_theme(style="whitegrid")
print("All libraries imported successfully!")

### Step 3: Load and Explore the Dataset

In [ ]:
# Step 3: Load the dataset into a pandas DataFrame
df = pd.read_csv("Churn_Modelling.csv")

# Display dataset dimensions and the first 5 records
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# View column information, non-null counts, and data types
df.info()

In [ ]:
# Numerical features statistical summary
df.describe()

### Step 4: Handle Missing Values and Incorrect Data Types

In [ ]:
# Step 4: Check for missing / null values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

### Step 5: Identify Target Column and Define Feature Sets
- **Target (`y`):** `Exited` (1 = Customer Left, 0 = Customer Retained)
- **Non-Predictive Columns:** `RowNumber`, `CustomerId`, `Surname` (dropped)
- **Categorical Features:** `Geography`, `Gender`
- **Numerical Features:** `CreditScore`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, `EstimatedSalary`

In [ ]:
# Step 5: Separate target column and feature columns
target_col = "Exited"
id_cols = ["RowNumber", "CustomerId", "Surname"]

# Drop identifier columns and target from features matrix X
X = df.drop(columns=id_cols + [target_col])
y = df[target_col]

# Check class balance of the target column
print("Target Class Distribution:")
print(y.value_counts(normalize=True).rename("Proportion"))

# Define column groups for transformation
numerical_cols = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary"
]
categorical_cols = ["Geography", "Gender"]

print(f"\nNumerical Features ({len(numerical_cols)}):", numerical_cols)
print(f"Categorical Features ({len(categorical_cols)}):", categorical_cols)

### Step 6: Train-Test Split with Stratification
Since only ~20% of customers churned, we use `stratify=y` to preserve the churn ratio in both the training set (80%) and test set (20%).

In [ ]:
# Step 6: Stratified split into train (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training samples: {X_train.shape[0]} ({y_train.mean():.2%} churn rate)")
print(f"Testing samples:  {X_test.shape[0]} ({y_test.mean():.2%} churn rate)")

### Step 7: Build Preprocessing Pipeline and Train Logistic Regression
We use `ColumnTransformer` to combine:
- `StandardScaler` for numerical attributes
- `OneHotEncoder(drop='first', handle_unknown='ignore')` for categorical attributes

We then package the preprocessor and `LogisticRegression(class_weight='balanced')` into a single Scikit-Learn `Pipeline`.

In [ ]:
# Step 7: Build ColumnTransformer and Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)
    ]
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

# Train pipeline on training split
model_pipeline.fit(X_train, y_train)
print("Model pipeline fitted successfully!")

### Step 8: Model Evaluation
Evaluate test set performance using Confusion Matrix, Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
# Step 8: Predictions and Performance Metrics
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

# Classification report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Retained (0)", "Churned (1)"]))

# ROC-AUC score
auc_val = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC Score: {auc_val:.4f}")
print("=" * 60)

In [ ]:
# Confusion Matrix and ROC Curve Visualizations
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Stay (0)", "Churn (1)"],
    yticklabels=["Stay (0)", "Churn (1)"],
    ax=axes[0]
)
axes[0].set_title("Confusion Matrix", fontsize=14)
axes[0].set_xlabel("Predicted Label", fontsize=12)
axes[0].set_ylabel("True Label", fontsize=12)

# 2. Receiver Operating Characteristic (ROC) Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC = {auc_val:.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="Chance")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate", fontsize=12)
axes[1].set_ylabel("True Positive Rate", fontsize=12)
axes[1].set_title("ROC Curve", fontsize=14)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

### Step 9: Save the Complete Pipeline

In [ ]:
# Step 9: Save the trained pipeline using joblib
model_path = "churn_model.pkl"
joblib.dump(model_pipeline, model_path)
print(f"Saved trained model pipeline to '{model_path}' successfully!")

### Step 10: Download the Model File (Google Colab)
Run this cell to download `churn_model.pkl` to your computer. Place it in the same directory as `app.py` to run the Streamlit app.

In [ ]:
# Step 10: Download model in Google Colab
try:
    from google.colab import files
    files.download(model_path)
    print(f"Downloading {model_path}...")
except ImportError:
    print(f"{model_path} is already saved locally in your directory.")